# SR-CorrNet 6-speaker separation — training notebook (internet OFF, GPU)

Fine-tunes the `shinuh/sr-corrnet-ss-1ch-wsj-var-2-5spk` checkpoint to **6 concurrent
speakers** on WSJ0-{2..6}mix, using the repo's own trainer.

**Inputs to attach:** the prep notebook's output (deps + repo + checkpoint + data).
Optionally also a previous version of *this* notebook's output to resume across sessions.

Interruption-safe: the engine checkpoints every epoch **and** every
`save_intra_epoch_steps` steps to `/kaggle/working/...`; rerunning the train cell
auto-resumes from the newest checkpoint. All training knobs live in the PARAMS cell.

Known limitation: the checkpoint is 8 kHz, so output is band-limited to 4 kHz —
fricatives sound muffled. Inherent to the base model; no 16 kHz variant exists.


In [ ]:
# ---- 1. environment ------------------------------------------------------------
import sys, os, glob, shutil, subprocess, torch

def find_input(marker):
    hits = glob.glob(f"/kaggle/input/*/{marker}")
    assert hits, f"no /kaggle/input/*/{marker} — attach the prep notebook's output"
    return os.path.dirname(hits[0])

AUX = find_input("deps")                       # prep-notebook output root
DATA = find_input("mix/wav")             # usually == AUX
REPO = "/kaggle/working/SR_CorrNet_SS"
# optional: previous version of THIS notebook's output, to resume across sessions
PREV_OUTPUT = ""                               # e.g. "/kaggle/input/<prev-train-run>"

sys.path.insert(0, f"{AUX}/deps")
# /kaggle/input is read-only; the repo resolves configs/logs inside its own tree -> copy out
if not os.path.exists(REPO):
    shutil.copytree(f"{AUX}/SR_CorrNet_SS", REPO)
sys.path.insert(0, REPO)

LOG_DIR = f"{REPO}/sr_corrnet/models/SR_CorrNet_SS/log/log_1ch_var_2_6spk/scratch_weights"
if PREV_OUTPUT and not os.path.exists(LOG_DIR):
    shutil.copytree(f"{PREV_OUTPUT}/SR_CorrNet_SS/sr_corrnet/models/SR_CorrNet_SS/log/log_1ch_var_2_6spk/scratch_weights", LOG_DIR)
    print("restored checkpoints from previous run")

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))


In [ ]:
# ---- 2. PARAMS — every training knob, edit here only ----------------------------
PARAMS = {
    # optimization
    "lr": 1.0e-4,                 # fine-tune LR (from-scratch recipe used 5e-4)
    "weight_decay": 1.0e-2,
    "warmup_steps": 500,          # iteration-based warmup, applied during epoch 1 only
    "clip_norm": 5,
    "scheduler_gamma": 0.95,      # StepLR decay per epoch, after start_scheduling
    "start_scheduling": 20,
    "max_epoch": 100,
    # data / loader
    "batch_size": 4,              # ~12 GB per sample at 4 s / 6 spk -> ~6 fits in 96 GB
    "num_workers": 4,
    "max_len": 32000,             # 4 s @ 8 kHz
    "num_per_epoch_train": 20000,
    "num_per_epoch_valid": 5000,
    "subset_dir": ["6mix", "5mix", "4mix", "3mix", "2mix"],
    # robustness
    "save_intra_epoch_steps": 2000,  # mid-epoch checkpoint cadence (steps)
}
CONFIG_NAME = "1ch_var_2_6spk.yaml"


In [ ]:
# ---- 3a. patch: O(N^2) pairwise PIT loss (verified == brute force, ~1200x faster at N=6)
loss_src = r'''import torch
import torch.nn as nn
import numpy as np

from math import ceil
from itertools import permutations
from dataclasses import dataclass, field, fields
from loguru import logger
from sr_corrnet.utils.decorators import logger_wraps
from sr_corrnet.utils import util_stft


# Utility functions
def l2norm(mat, keepdim=False):
    return torch.norm(mat, dim=-1, keepdim=keepdim)

def l1norm(mat, keepdim=False):
    return torch.norm(mat, dim=-1, keepdim=keepdim, p=1)


       
@logger_wraps()
class PIT_SISNR_mag(nn.Module):
    def __init__(self, scale_inv: bool, device: torch.device):
        super().__init__()
        self.device = device
        self.scale_inv = scale_inv
        self.stft = util_stft.STFT(frame_length=512, frame_shift=256, device=self.device, normalize=True)

    def forward(self, estims, targets, eps=1.0e-10, prior_idx=None):
        assert len(estims) == len(targets), "The number of estimated sources and target sources must be the same."
        n_spks = len(estims)

        def _SDR_loss(permute, batch_idx=None):
            loss_for_permute = []
            for s, t in enumerate(permute):
                est = estims[s]
                src = targets[t]

                # If batch_idx is provided, select specific batch samples
                if batch_idx is not None:
                    est = est[[batch_idx]]
                    src = src[[batch_idx]]

                est_zm = est - torch.mean(input=est, dim=-1, keepdim=True)
                src_zm = src - torch.mean(input=src, dim=-1, keepdim=True)

                if self.scale_inv:
                    scale_factor = torch.sum(est * src, dim=-1, keepdim=True) / (l2norm(l2norm(src, keepdim=True),keepdim=True)**2 + eps)
                    src_zm_scale = scale_factor * src_zm

                est_stft = self.stft(est_zm, cplx=True)
                src_stft = self.stft(src_zm_scale, cplx=True)
                est_mag = torch.sqrt(est_stft.real**2 + est_stft.imag**2)
                src_mag = torch.sqrt(src_stft.real**2 + src_stft.imag**2)

                utt_loss = - 20 * torch.log10(eps + l2norm(l2norm(src_mag)) / (l2norm(l2norm(est_mag - src_mag)) + eps))
                utt_loss = torch.clamp(utt_loss, min=-30)
                
                loss_for_permute.append(utt_loss)
            return sum(loss_for_permute)/n_spks
        if prior_idx is not None:
            # Handle both single permutation and batch-wise permutations
            if isinstance(prior_idx, list) and len(prior_idx) > 0 and isinstance(prior_idx[0], list):
                # Batch-wise permutations: compute loss for each batch with its specific permutation
                batch_losses = [_SDR_loss(perm, batch_idx=b) for b, perm in enumerate(prior_idx)]
                min_perutt = torch.stack(batch_losses)
            else:
                # Single permutation for all batches
                min_perutt = _SDR_loss(prior_idx)
        else:
            pscore = torch.cat([_SDR_loss(p) for p in permutations(range(n_spks))])
            min_perutt, _ = torch.min(pscore, dim=0)

        return torch.mean(min_perutt)


@logger_wraps()
class PIT_SISNR_time(nn.Module):
    def __init__(self, scale_inv: bool, device: torch.device):
        super().__init__()
        self.device = device
        self.scale_inv = scale_inv

    def forward(self, estims, targets, eps=1.0e-10, return_perm_idx=False):
        assert len(estims) == len(targets), "The number of estimated sources and target sources must be the same."
        n_spks = len(estims)
        # Pairwise [n_est, n_ref] SI-SNR matrix computed once; permutations are then
        # scored by indexing into it. O(N^2) heavy work instead of O(N!*N).
        est = torch.stack(list(estims), dim=0)      # [n, ..., T]
        src = torch.stack(list(targets), dim=0)     # [n, ..., T]
        est_zm = est - torch.mean(est, dim=-1, keepdim=True)
        src_zm = src - torch.mean(src, dim=-1, keepdim=True)
        e = est_zm.unsqueeze(1)                     # [n, 1, ..., T]
        r = src_zm.unsqueeze(0)                     # [1, n, ..., T]
        r_scale = r
        if self.scale_inv:
            scale_factor = torch.sum(e * r, dim=-1, keepdim=True) / (l2norm(r, keepdim=True)**2 + eps)
            r_scale = scale_factor * r
        pair_loss = - 20 * torch.log10(eps + l2norm(r_scale) / (l2norm(e - r_scale) + eps))
        pair_loss = torch.clamp(pair_loss, min=-30)  # [n_est, n_ref, ...]

        perms = torch.tensor(list(permutations(range(n_spks))), device=pair_loss.device)  # [n!, n]
        rows = torch.arange(n_spks, device=pair_loss.device).unsqueeze(0).expand_as(perms)
        pscore = pair_loss[rows, perms].mean(dim=1)  # [n!, ...]
        indices = [list(p) for p in permutations(range(n_spks))]
        min_perutt, min_idx = torch.min(pscore, dim=0)
        if return_perm_idx:
            # Handle batch-wise indexing for permutation indices
            if min_idx.dim() > 0:  # batch size > 1
                batch_indices = [indices[idx.item()] for idx in min_idx]
                return torch.mean(min_perutt), batch_indices
            else:  # batch size = 1
                return torch.mean(min_perutt), indices[min_idx.item()]
        else:
            return torch.mean(min_perutt)


@logger_wraps()
class PIT_SISNRi(nn.Module):
    def __init__(self, scale_inv: bool, device: torch.device):
        super().__init__()
        self.device = device
        self.scale_inv = scale_inv
    
    def forward(self, estims, targets, input, eps=1.0e-20):
        assert len(estims) == len(targets), "The number of estimated sources and target sources must be the same."
        n_spks = len(estims)
        input_zm = input - torch.mean(input, dim=-1, keepdim=True)

        # Same pairwise-matrix trick as PIT_SISNR_time. The mixture term depends only
        # on the reference index, so summed over any full permutation it is constant.
        est = torch.stack(list(estims), dim=0)      # [n, ..., T]
        src = torch.stack(list(targets), dim=0)     # [n, ..., T]
        est_zm = est - torch.mean(est, dim=-1, keepdim=True)
        src_zm = src - torch.mean(src, dim=-1, keepdim=True)
        e = est_zm.unsqueeze(1)                     # [n, 1, ..., T]
        r = src_zm.unsqueeze(0)                     # [1, n, ..., T]
        r_s = r
        if self.scale_inv:
            factor = torch.sum(e * r, dim=-1, keepdim=True) / (l2norm(r, keepdim=True)**2 + eps)
            r_s = factor * r
        pair_est = 20 * torch.log10(eps + l2norm(r_s) / (l2norm(e - r_s) + eps))  # [n_est, n_ref, ...]

        src_zm_x = src_zm
        if self.scale_inv:
            src_zm_x = torch.sum(input_zm.unsqueeze(0) * src_zm, dim=-1, keepdim=True) / (l2norm(src_zm, keepdim=True)**2 + eps) * src_zm
        loss_in = 20 * torch.log10(eps + l2norm(src_zm_x) / (l2norm(input_zm.unsqueeze(0) - src_zm_x) + eps))  # [n_ref, ...]

        perms = torch.tensor(list(permutations(range(n_spks))), device=pair_est.device)  # [n!, n]
        rows = torch.arange(n_spks, device=pair_est.device).unsqueeze(0).expand_as(perms)
        pscore = pair_est[rows, perms].mean(dim=1) - loss_in.mean(dim=0)  # [n!, ...]
        max_perutt, max_idx = torch.max(pscore, dim=0)
        return torch.mean(max_perutt)
'''
with open('{REPO}/sr_corrnet/models/SR_CorrNet_SS/loss.py'.format(REPO=REPO), 'w', encoding='utf-8') as f:
    f.write(loss_src)
print('loss.py patched')


In [ ]:
# ---- 3b. patch: intra-epoch checkpointing in the train loop
ANCHOR = '''            # update the progress bar
            torch.cuda.synchronize()
            n_batch += 1
            pbar.update(1)'''
NEW = '''            # update the progress bar
            torch.cuda.synchronize()
            n_batch += 1
            # periodic intra-epoch save so an interrupted session loses minutes, not the
            # whole epoch; the end-of-epoch save overwrites the same file
            save_every = self.config['engine'].get('save_intra_epoch_steps', 0)
            if save_every and n_batch % save_every == 0:
                torch.save({"epoch": epoch,
                            "model_state_dict": self.model.state_dict(),
                            "optimizer_state_dict": self.main_optimizer.state_dict()},
                           os.path.join(self.checkpoint_path, f"epoch.{epoch:04}.pth"))
            pbar.update(1)'''
eng_path = f"{REPO}/sr_corrnet/models/SR_CorrNet_SS/engine.py"
src = open(eng_path).read()
if "save_intra_epoch_steps" in src:
    print("engine.py already patched")
else:
    assert ANCHOR in src, "engine.py changed upstream - re-derive the patch"
    open(eng_path, "w").write(src.replace(ANCHOR, NEW))
    print("engine.py patched")


In [ ]:
# ---- 3c. write the 6-speaker config, applying PARAMS
import yaml
cfg_template = r'''project: "[Project] SR_CorrNet_SS" ### Dont't change
notes: "Fine-tune of var-2-5spk checkpoint to WSJ0-{2..6}mix with Attractor-based Split (max_n_spks 5->6)"
# ------------------------------------------------------------------------------------------------------------------------------ #
config:
    # ⚡⚡⚡⚡⚡⚡⚡⚡⚡ key variables ⚡⚡⚡⚡⚡⚡⚡⚡⚡ #
    test_unknown_n_spks: true
    max_n_spks: &var_model_n_spks 6
    is_var_spks: &var_is_var_spks true

    num_mics: &var_model_num_mics 1
    taps_freq: &var_taps_freq [1,1] # [Future, Past]
    taps_frame: &var_taps [1,1] # [Future, Past]

    N_Enc: &var_num_enc 2 # separation encoder
    N_Dec: &var_num_dec 4 # reconstruction decoder

    d_model: &var_model_channels 128
    d_hidden: &var_hidden_channels 384
    d_hidden_spk: &var_hidden_channels_spk 512
    n_head: &var_n_head 4
    kernel_size: &var_kernel_size 3
    # ------------------------------------------------------------ #
    eval_SDRi: false
    partitions:
        train: ["train", "valid"]
        test: ["test"]
    dataset:
        max_n_spks: *var_model_n_spks
        synthesis_config:
            max_len : 32000
            sampling_rate: 8000
        ref_ch: &var_ref_ch 0
        # NOTE: repo's dynamic mixing only synthesizes 2-speaker mixtures (dataset.py
        # _dynamic_mixing picks exactly one other key) — unusable for 6mix. Keep false.
        dynamic_mixing: false
        speed_list_for_DM: [0.95, 0.96, 0.97, 0.98, 0.99,
                            1.00, 1.00, 1.00, 1.00, 1.00,
                            1.01, 1.02, 1.03, 1.04, 1.05]
        scp_dir: "data/scp/scp_wsj0_mix_8k"
        subset_conf:
            train:
                subset: true
                num_per_epoch: 20000 # number of data samples for one epoch
            valid:
                subset: true
                num_per_epoch: 5000 # number of data samples for one epoch
        # Sampling is uniform over the pooled subsets, so the 6-vs-5-vs-... weighting
        # is set by how many mixtures of each count are generated on disk.
        subset_dir: ['6mix', '5mix', '4mix', '3mix', '2mix']
        2mix:
            train:
                mixture: "tr_mix.scp"
                spk: ["tr_s1.scp", "tr_s2.scp"]
            valid:
                mixture: "cv_mix.scp"
                spk: ["cv_s1.scp", "cv_s2.scp"]
            test:
                mixture: "tt_mix.scp"
                spk: ["tt_s1.scp", "tt_s2.scp"]
        3mix:
            train:
                mixture: "tr_mix.scp"
                spk: ["tr_s1.scp", "tr_s2.scp", "tr_s3.scp"]
            valid:
                mixture: "cv_mix.scp"
                spk: ["cv_s1.scp", "cv_s2.scp", "cv_s3.scp"]
            test:
                mixture: "tt_mix.scp"
                spk: ["tt_s1.scp", "tt_s2.scp", "tt_s3.scp"]
        4mix:
            train:
                mixture: "tr_mix.scp"
                spk: ["tr_s1.scp", "tr_s2.scp", "tr_s3.scp", "tr_s4.scp"]
            valid:
                mixture: "cv_mix.scp"
                spk: ["cv_s1.scp", "cv_s2.scp", "cv_s3.scp", "cv_s4.scp"]
            test:
                mixture: "tt_mix.scp"
                spk: ["tt_s1.scp", "tt_s2.scp", "tt_s3.scp", "tt_s4.scp"]
        5mix:
            train:
                mixture: "tr_mix.scp"
                spk: ["tr_s1.scp", "tr_s2.scp", "tr_s3.scp", "tr_s4.scp", "tr_s5.scp"]
            valid:
                mixture: "cv_mix.scp"
                spk: ["cv_s1.scp", "cv_s2.scp", "cv_s3.scp", "cv_s4.scp", "cv_s5.scp"]
            test:
                mixture: "tt_mix.scp"
                spk: ["tt_s1.scp", "tt_s2.scp", "tt_s3.scp", "tt_s4.scp", "tt_s5.scp"]
        6mix:
            train:
                mixture: "tr_mix.scp"
                spk: ["tr_s1.scp", "tr_s2.scp", "tr_s3.scp", "tr_s4.scp", "tr_s5.scp", "tr_s6.scp"]
            valid:
                mixture: "cv_mix.scp"
                spk: ["cv_s1.scp", "cv_s2.scp", "cv_s3.scp", "cv_s4.scp", "cv_s5.scp", "cv_s6.scp"]
            test:
                mixture: "tt_mix.scp"
                spk: ["tt_s1.scp", "tt_s2.scp", "tt_s3.scp", "tt_s4.scp", "tt_s5.scp", "tt_s6.scp"]
    # ------------------------------------------------------------ #
    dataloader:
        pin_memory: true
        batch_size: 1
        drop_last: false
        num_workers: 4
    # ------------------------------------------------------------ #
    stft:
        frame_length: 128
        frame_shift: 64
    # ------------------------------------------------------------ #
    model:
        ref_ch: *var_ref_ch
        taps_freq: *var_taps_freq
        taps_frame: *var_taps
        d_model: *var_model_channels
        input_embedding:
            d_model: *var_model_channels
            kernel_sizes: [3, 3] # first layer, second layer
            num_mics: *var_model_num_mics
            taps_freq: *var_taps_freq
            taps_frame: *var_taps
        RoPE:
            d_model: *var_model_channels
            n_head: *var_n_head
        multi_path_block:
            channel_module:
                d_model: *var_model_channels
                d_hidden: *var_hidden_channels
                n_head: *var_n_head
                kernel_size: *var_kernel_size
                dropout_rate: 0.00
            spectral_module:
                d_model: *var_model_channels
                d_hidden: *var_hidden_channels
                n_head: *var_n_head
                kernel_size: *var_kernel_size
                dropout_rate: 0.00
        is_var_spks: *var_is_var_spks
        spk_split:
            varying:
                d_model: *var_model_channels
                d_freq: &num_frequency_bin 65 # win/2+1
                n_head: *var_n_head
                max_n_spks: *var_model_n_spks
                dropout_rate: 0.00
            fixed:
                d_model: *var_model_channels
                max_n_spks: *var_model_n_spks
        cross_spk:
            d_model: *var_model_channels
            d_hidden: *var_hidden_channels_spk
            n_head: *var_n_head
            dropout_rate: 0.00
        N_Enc: *var_num_enc
        N_Dec: *var_num_dec
        mask_estimator:
            d_model: *var_model_channels
            num_mics: *var_model_num_mics
            taps_freq: *var_taps_freq
            taps_frame: *var_taps
            MIMO: &var_mimo_opt false
        apply_pos_enc: true
    # ------------------------------------------------------------ #
    engine:
        # -------------------------------- #
        optimizer:
            name: "AdamW" ### Choose a torch.optim's class(=attribute) e.g. ["Adam", "AdamW", "SGD", ...]
            AdamW:
                # fine-tune from a trained checkpoint: 5x lower than the from-scratch 5.0e-4
                lr: 1.0e-4
                weight_decay: 1.0e-2
            Adam:
                lr: 1.0e-4
                weight_decay : 0.0
        # -------------------------------- #
        scheduler:
            name: "StepLR" # ReduceLROnPlateau,
            WarmupConstantSchedule:
                warmup_steps: 500
            ReduceLROnPlateau:
                mode: "min"
                min_lr: 1.0e-10
                factor: 0.5
                patience: 2
            StepLR:
                step_size: 1
                gamma: 0.95
        # -------------------------------- #
        check_compute_len: [2, 65, 125]
        max_epoch: 100
        clip_norm: 5
        save_intra_epoch_steps: 2000
        start_scheduling: 20


'''
doc = yaml.safe_load(cfg_template)
c = doc["config"]
c["engine"]["optimizer"]["AdamW"].update(lr=PARAMS["lr"], weight_decay=PARAMS["weight_decay"])
c["engine"]["scheduler"]["WarmupConstantSchedule"]["warmup_steps"] = PARAMS["warmup_steps"]
c["engine"]["scheduler"]["StepLR"]["gamma"] = PARAMS["scheduler_gamma"]
c["engine"].update(max_epoch=PARAMS["max_epoch"], clip_norm=PARAMS["clip_norm"],
                   start_scheduling=PARAMS["start_scheduling"],
                   save_intra_epoch_steps=PARAMS["save_intra_epoch_steps"])
c["dataloader"].update(batch_size=PARAMS["batch_size"], num_workers=PARAMS["num_workers"])
c["dataset"]["synthesis_config"]["max_len"] = PARAMS["max_len"]
c["dataset"]["subset_conf"]["train"]["num_per_epoch"] = PARAMS["num_per_epoch_train"]
c["dataset"]["subset_conf"]["valid"]["num_per_epoch"] = PARAMS["num_per_epoch_valid"]
c["dataset"]["subset_dir"] = PARAMS["subset_dir"]
c["dataset"]["scp_dir"] = "/kaggle/working/scp_mix_8k"
with open(f"{REPO}/sr_corrnet/models/SR_CorrNet_SS/configs/{CONFIG_NAME}", "w") as f:
    yaml.dump(doc, f, sort_keys=False)
print("config written:", CONFIG_NAME)


In [ ]:
# ---- 4. regenerate SCPs against the read-only data mount -------------------------
# (the prep notebook recorded absolute paths valid at generation time; keys = file stems)
part_map = {"tr": "tr", "cv": "cv", "tt": "tt"}
for sub in PARAMS["subset_dir"]:
    n = int(sub[0])
    for part in ["tr", "cv", "tt"]:
        src_dir = f"{DATA}/mix/wav/{sub}/{part}"
        if not os.path.isdir(src_dir):
            print(f"missing {src_dir} - skipped"); continue
        out = f"/kaggle/working/scp_mix_8k/{sub}"
        os.makedirs(out, exist_ok=True)
        for d in ["mix"] + [f"s{k}" for k in range(1, n + 1)]:
            wavs = sorted(glob.glob(f"{src_dir}/{d}/*.wav"))
            assert wavs, f"no wavs in {src_dir}/{d}"
            with open(f"{out}/{part}_{'mix' if d == 'mix' else d}.scp", "w") as f:
                f.writelines(f"{os.path.splitext(os.path.basename(w))[0]} {w}\n" for w in wavs)
    print(sub, "scps written")


In [ ]:
# ---- 5. model prep: widen spk_query 5->6 and seed the resume checkpoint ----------
# engine resume is file-existence based: newest epoch.NNNN.pth in LOG_DIR wins.
# Idempotent: skipped whenever any checkpoint (seed or later) already exists.
if glob.glob(f"{LOG_DIR}/epoch.*.pth"):
    print("checkpoints exist - training will resume:", sorted(os.listdir(LOG_DIR))[-3:])
else:
    sd = torch.load(f"{AUX}/ckpt/model.pt", map_location="cpu", weights_only=True)
    sd = {k: v for k, v in sd.items() if not k.endswith(("total_ops", "total_params"))}
    # spk_query: (1, max_n_spks+2, d) - slot 0 aux, 1..5 speakers, last residual.
    # New speaker row at index 6 = mean of speaker rows + small noise; residual stays last.
    q = sd["spk_split.spk_query"]; assert tuple(q.shape) == (1, 7, 128), q.shape
    torch.manual_seed(0)
    spk = q[:, 1:6]
    new_row = spk.mean(dim=1, keepdim=True) + 0.1 * spk.std() * torch.randn(1, 1, q.shape[-1])
    sd["spk_split.spk_query"] = torch.cat([q[:, :6], new_row, q[:, 6:]], dim=1)

    import yaml
    cfg = yaml.safe_load(open(f"{REPO}/sr_corrnet/models/SR_CorrNet_SS/configs/{CONFIG_NAME}"))["config"]
    from sr_corrnet.models.SR_CorrNet_SS.model import Model
    model = Model(**cfg["model"])
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, (missing, unexpected)
    opt = torch.optim.AdamW(model.parameters(), lr=PARAMS["lr"], weight_decay=PARAMS["weight_decay"])
    os.makedirs(LOG_DIR, exist_ok=True)
    torch.save({"epoch": 0, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": opt.state_dict()}, f"{LOG_DIR}/epoch.0000.pth")
    print("seeded 6-spk warm-start checkpoint, params:", sum(p.numel() for p in model.parameters()))


In [ ]:
# ---- 6. train (rerun this cell to resume after any interruption) -----------------
env = dict(os.environ, PYTHONPATH=f"{AUX}/deps:{REPO}", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
subprocess.run([sys.executable, "run.py", "--model", "SR_CorrNet_SS", "--engine_mode", "train",
                "--config", CONFIG_NAME, "--gpuid", "0"], cwd=REPO, env=env, check=True)


In [ ]:
# ---- 7. evaluate: SI-SNRi per speaker count (uses best exported checkpoint) ------
env = dict(os.environ, PYTHONPATH=f"{AUX}/deps:{REPO}", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
subprocess.run([sys.executable, "run.py", "--model", "SR_CorrNet_SS", "--engine_mode", "test",
                "--config", CONFIG_NAME, "--gpuid", "0"], cwd=REPO, env=env, check=True)
# per-count results are logged as EVAL-TEST_6MIX ... EVAL-TEST_2MIX above


In [ ]:
# ---- 8. demo: mixture in -> per-speaker wavs out ----------------------------------
from sr_corrnet.inference import SSInference
from IPython.display import Audio, display

demo_mix = sorted(glob.glob(f"{DATA}/mix/wav/6mix/tt/mix/*.wav"))[0]
m = SSInference.from_pretrained(config=CONFIG_NAME,
                                checkpoint_path=f"{REPO}/sr_corrnet/checkpoints/SS/1ch_var_2_6spk/model.pt",
                                device="cuda:0")
m.process_file(demo_mix, output_dir="/kaggle/working/demo_out")
print("input:", demo_mix)
display(Audio(demo_mix))
for w in sorted(glob.glob("/kaggle/working/demo_out/*.wav")):
    print(w); display(Audio(w))
